In [ ]:
import time
import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 글로벌 시드 고정
np.random.seed(42)

def evaluate_ensemble_variance_floor():
    """ [주제 4] Bagging vs Pasting: 중복 허용이 앙상블 분산 공식의 상관계수(ρ)를 낮추는 원리 """
    print("=" * 75)
    print("[주제 4] 복원(Bagging) vs 비복원(Pasting) 추출에 따른 트리 간 동질성(ρ) 격차 실측")
    print("=" * 75)

    X, y = make_classification(n_samples=1500, n_features=20, random_state=42)

    # 1. Pasting (비복원 추출: 개별 트리가 전체 트렌드를 공유함)
    pasting_model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, bootstrap=False, random_state=42).fit(X, y)
    # 2. Bagging (복원 추출: 중복 샘플링을 통한 데이터 공간의 강제 왜곡)
    bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, bootstrap=True, random_state=42).fit(X, y)

    # 각 베이스 트리들의 예측 행렬 도출
    pasting_preds = np.array([est.predict(X) for est in pasting_model.estimators_])
    bagging_preds = np.array([est.predict(X) for est in bagging_model.estimators_])

    # 통계 공식의 트리 간 상관계수(rho) 평균값 산출
    rho_pasting = np.mean(np.corrcoef(pasting_preds))
    rho_bagging = np.mean(np.corrcoef(bagging_preds))

    print(f" ├ 1. 비복원 추출(Pasting) 트리 간 평균 상관계수 (ρ) : {rho_pasting:.4f}")
    print(f" ├ 2. 복원 추출(Bagging)  트리 간 평균 상관계수 (ρ) : {rho_bagging:.4f}")
    print(f" └ 진단: 배깅의 중복 추출이 트리의 독립적 에러 패턴을 형성(De-correlation)하여 분산 하한선을 낮춤.")


def benchmark_modern_boosting_engines():
    """ [주제 7] 현대 하이테크 부스팅 실측: XGBoost vs LightGBM """
    print("\n" + "=" * 75)
    print("[주제 7] 분할 알고리즘 혁신(Level-wise vs Leaf-wise)에 따른 연산 시간 격차")
    print("=" * 75)

    # 연산 스트레스를 위한 대용량 가상 회귀 데이터셋 구축
    X_boost, y_boost = make_regression(n_samples=40000, n_features=50, random_state=42)

    # 1. XGBoost (Level-wise 수평 정렬 분할)
    xgb = XGBRegressor(n_estimators=100, max_depth=6, n_jobs=-1, random_state=42)
    start = time.time()
    xgb.fit(X_boost, y_boost)
    time_xgb = time.time() - start
    print(f" ├ XGBoost 학습 소요 시간 (Level-wise & Hessian)   : {time_xgb:.4f} 초")

    # 2. LightGBM (Leaf-wise 수직 최적 분할)
    lgb = LGBMRegressor(n_estimators=100, max_depth=6, n_jobs=-1, random_state=42, verbose=-1)
    start = time.time()
    lgb.fit(X_boost, y_boost)
    time_lgb = time.time() - start
    print(f" ├ LightGBM 학습 소요 시간 (Leaf-wise & GOSS)     : {time_lgb:.4f} 초")

    speedup = time_xgb / time_lgb
    print(f" └ 결론: LightGBM이 비대칭 리프 중심 분할 구조를 통해 연산 효율성을 약 {speedup:.1f}배 혁신함.")

if __name__ == "__main__":
    evaluate_ensemble_variance_floor()
    benchmark_modern_boosting_engines()